In [10]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')

text = """Langchain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

sentences = [s for s in text.split('\n') if s.strip() != '']

embeddings = model.encode(sentences)

threshold = 0.7

chunks  = []

current_chunk = [sentences[0]]

for i in range(1, len(sentences)):
    cosine = cosine_similarity([embeddings[i]], [embeddings[i-1]])[0][0]

    if cosine >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(' '.join(current_chunk))
        current_chunk = [sentences[i]]


chunks.append(' '.join(current_chunk))

for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx+1}:\n{chunk}\n")
    print("-----")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6917.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunk 1:
Langchain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

-----
Chunk 2:
You can create chains, agents, memory, and retrievers.

-----
Chunk 3:
The Eiffel Tower is located in Paris.

-----
Chunk 4:
France is a popular tourist destination.

-----


In [12]:
from langchain_classic.schema import Document
from langchain_classic.vectorstores import FAISS
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.schema.runnable import RunnableLambda, RunnableMap
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [13]:
class ThresholdSemanticChunker:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2', threshold: float = 0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def chunk(self, text: str) -> list[str]:
        sentences = [s for s in text.split('\n') if s.strip() != '']
        embeddings = self.model.encode(sentences)

        chunks  = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            cosine = cosine_similarity([embeddings[i]], [embeddings[i-1]])[0][0]

            if cosine >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(' '.join(current_chunk))
                current_chunk = [sentences[i]]

        chunks.append(' '.join(current_chunk))
        return chunks

    def split_documents(self, documents: list[Document]) -> list[Document]:
        split_docs = []
        for doc in documents:
            chunks = self.chunk(doc.page_content)
            for chunk in chunks:
                split_docs.append(Document(page_content=chunk, metadata=doc.metadata))
        return split_docs

In [14]:
sample_text = """Langchain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

doc = Document(page_content=sample_text)

chunker = ThresholdSemanticChunker()

chunks = chunker.split_documents([doc])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19919.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
chunks

[Document(metadata={}, page_content='Langchain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [16]:
vectorstore = FAISS.from_documents(chunks, HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2'))
retriever  = vectorstore.as_retriever()



/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/ipykernel_31829/869379678.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  vectorstore = FAISS.from_documents(chunks, HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2'))
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11421.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
template  = """You are a helpful assistant. Use the following retrieved context to answer the question.
{context}
Question: {question}
Answer:"""

prompt = PromptTemplate.from_template(template)

In [18]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.7)

rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x['question']),
            "question": lambda x: x['question']
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

query = {"question": "What is langchain?"}
result = rag_chain.invoke(query)
print(result)

Langchain is a framework for building applications with LLMs. It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers using Langchain.


## Semantic Chunking in Langchain

In [19]:

from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_classic.document_loaders import TextLoader



In [21]:
loader = TextLoader('/Users/pawanpahune/RAG_From_Scratch/Semantic_Chunking/langchain.txt')



In [23]:
loader.load()

[Document(metadata={'source': '/Users/pawanpahune/RAG_From_Scratch/Semantic_Chunking/langchain.txt'}, page_content='Langchain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.')]

In [25]:
embedding = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
chunker = SemanticChunker(embedding)
chunks = chunker.split_documents(loader.load())

for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx+1}:\n{chunk.page_content}\n")
    print("-----")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5954.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunk 1:
Langchain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

-----
Chunk 2:
The Eiffel Tower is located in Paris. France is a popular tourist destination.

-----


In [26]:
new_document = """reinforcement learning is a type of machine learning where an agent learns to make decisions by taking actions in an environment to maximize cumulative reward. 
It is inspired by behavioral psychology and is used in various applications such as robotics, game playing, and autonomous systems. 
the agent receives feedback in the form of rewards or penalties based on its actions, and it uses this feedback to improve its decision-making over time. 
Reinforcement learning algorithms include Q-learning, deep Q-networks (DQN), and policy gradient methods.
RL has successfully been applied to complex tasks like playing Go, controlling robots, and optimizing resource management. 
It continues to be an active area of research with ongoing advancements in algorithms and applications."""

In [27]:
new_doc = Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [28]:
new_doc


Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='reinforcement learning is a type of machine learning where an agent learns to make decisions by taking actions in an environment to maximize cumulative reward. \nIt is inspired by behavioral psychology and is used in various applications such as robotics, game playing, and autonomous systems. \nthe agent receives feedback in the form of rewards or penalties based on its actions, and it uses this feedback to improve its decision-making over time. \nReinforcement learning algorithms include Q-learning, deep Q-networks (DQN), and policy gradient methods.\nRL has successfully been applied to complex tasks like playing Go, controlling robots, and optimizing resource management. \nIt continues to be an active area of research with ongoing advancements in algorithms and applications.')

In [29]:
vectorstore

In [30]:
new_chunks = chunker.split_documents([new_doc])

In [31]:
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='reinforcement learning is a type of machine learning where an agent learns to make decisions by taking actions in an environment to maximize cumulative reward. It is inspired by behavioral psychology and is used in various applications such as robotics, game playing, and autonomous systems. the agent receives feedback in the form of rewards or penalties based on its actions, and it uses this feedback to improve its decision-making over time. Reinforcement learning algorithms include Q-learning, deep Q-networks (DQN), and policy gradient methods. RL has successfully been applied to complex tasks like playing Go, controlling robots, and optimizing resource management.'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='It continues to be an active area of research with ongoing advancements in algorithms and applications.')]

In [33]:
vectorstore.add_documents(new_chunks)

['be77dbd8-b2b4-49d7-a2e2-c8c3cca9453e',
 '2ca85af4-f869-4479-894e-d814eef8ac97']

In [35]:
retriever  = vectorstore.as_retriever()

In [37]:
new_question = "what are key concepts in reinforcement learning?"

rag_chain.invoke({"question": new_question})


'Key concepts in reinforcement learning include:\n\n*   **Agent:** An entity that learns to make decisions.\n*   **Environment:** The setting where the agent takes actions.\n*   **Actions:** The choices made by the agent within the environment.\n*   **Rewards or Penalties (Feedback):** Signals received by the agent based on its actions, used to improve decision-making.\n*   **Cumulative Reward:** The ultimate goal for the agent to maximize over time.\n*   **Decision-making improvement:** The process by which the agent learns from feedback to make better choices over time.\n*   **Algorithms:** Specific methods used in RL, such as Q-learning, Deep Q-Networks (DQN), and policy gradient methods.\n\nReinforcement learning is also inspired by behavioral psychology.'